# v4.3-lite — model definition & smoke test

Slimmed hybrid CNN+GNN classifier built to combat the overfitting seen in v4.2/v4.3. Differences vs the v4.3 trunk are summarised at the top of each section.

**Goals of this notebook**
1. Define the *lite* model end-to-end in clean cells.
2. Verify the parameter count is in the target range (~0.4–0.5 M, vs ~1.9 M for v4.3).
3. Run a dummy-batch forward pass to catch shape bugs *before* touching real data.

Once both checks pass, move to `02_train.ipynb` for the data pipeline + training loop.

## 1. Configuration

Hyperparameters live here so they can be reused unchanged in `02_train.ipynb` and `03_eval_and_compare.ipynb`. *Lite-vs-v4.3* changes are flagged in the comments.

In [1]:
from typing import Tuple

# ---- Sequence canvas ----
FIXED_LENGTH = 20_000              # unchanged from v4.3

# ---- CNN tower (lite) ----
CNN_WIDTH        = 64              # v4.3: 128 -> 64 (lever A: capacity)
MOTIF_KERNELS    = (7, 15)         # v4.3: (7, 15, 21) -> drop 21 (lever A)
CONTEXT_KERNEL   = 9               # unchanged
CONTEXT_DILATIONS = (1, 2, 4)      # v4.3: (1, 2, 4, 8) -> drop the dilation=8 block (lever A)
POS_ENC_CHANNELS = 4               # carried from v4.4: 4 sinusoidal channels appended to one-hot
CNN_DROPOUT      = 0.30            # v4.3: 0.15 -> 0.30 (lever B: regularisation)
RC_FUSION_MODE   = 'late'          # keep late-fusion test-time RC averaging only

# ---- GNN tower (lite) ----
KMER_K        = 6                  # v4.3: 7 -> 6 (lever A: shrink k-mer hash space)
KMER_DIM      = 1024               # v4.3: 2048 -> 1024 (lever A)
KMER_WINDOW   = 512                # unchanged
KMER_STRIDE   = 256                # unchanged
GNN_HIDDEN    = 64                 # v4.3: 128 -> 64 (lever A)
GNN_LAYERS    = 2                  # v4.3: 3 -> 2 (lever A)
GNN_DROPOUT   = 0.25               # v4.3: 0.10 -> 0.25 (lever B)
GNN_IN_DIM    = KMER_DIM + 1       # +1 for normalised window-centre position

# ---- Fusion (lite) ----
FUSION_DIM    = 128                # v4.3: 256 -> 128 (lever A)
NUM_HEADS     = 2                  # v4.3: 4 -> 2 (lever A)
FUSION_DROPOUT = 0.30              # v4.3: 0.20 -> 0.30 (lever B)

# ---- Heads ----
# Set in 02_train.ipynb based on variant; placeholder values here for the smoke test.
DEFAULT_NUM_TOPLEVEL = 3           # 3 for three_class_*, 2 for binary_dna
DEFAULT_NUM_SUPERFAMILIES = 23     # ballpark for the v4.3 dataset

# ---- Smoke-test batch ----
SMOKE_BATCH = 4
SMOKE_NODES_PER_GRAPH = 12

print('Config loaded.')

Config loaded.


## 2. Imports & device

In [2]:
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
np.random.seed(0)

def resolve_device():
    if torch.cuda.is_available():
        return torch.device('cuda')
    if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
        return torch.device('mps')
    return torch.device('cpu')

DEVICE = resolve_device()
print(f'PyTorch {torch.__version__} on {DEVICE}')

# Reverse-complement permutation for one-hot channels A,C,G,T,N -> T,G,C,A,N
REV_COMP = torch.tensor([3, 2, 1, 0, 4], dtype=torch.long)

PyTorch 2.9.0 on mps


## 3. CNN building blocks

Carried from v4.3 verbatim because they are already minimal. The slim-down happens via the configuration in section 1 (fewer kernels, fewer dilated blocks, narrower channels).

In [3]:
class ConvBlock(nn.Module):
    """Residual dilated 1D conv block: Conv -> BN -> GELU -> Dropout (+ residual)."""

    def __init__(self, c_in: int, c_out: int, kernel_size: int = 9, dilation: int = 1, dropout: float = 0.1):
        super().__init__()
        assert kernel_size % 2 == 1, 'kernel_size must be odd'
        pad = (kernel_size // 2) * dilation
        self.conv = nn.Conv1d(c_in, c_out, kernel_size, padding=pad, dilation=dilation, bias=True)
        self.bn = nn.BatchNorm1d(c_out)
        self.drop = nn.Dropout1d(dropout)            # v4.3 used nn.Dropout (per-element); Dropout1d zeroes channels
        self.proj = nn.Identity() if c_in == c_out else nn.Conv1d(c_in, c_out, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        y = F.gelu(self.bn(self.conv(x)))
        y = self.drop(y)
        return y + self.proj(x)


class MaskedMaxPool1d(nn.Module):
    """Stride-2 max pool that respects a (B, L) padding mask."""

    def __init__(self, kernel_size: int = 2, stride: int = 2):
        super().__init__()
        self.kernel_size = kernel_size
        self.stride = stride

    def forward(self, x: torch.Tensor, mask: torch.Tensor):
        if mask is not None:
            m = mask.unsqueeze(1).float()
            x = x * m + (~mask.unsqueeze(1)) * (-1e9)
        x_p = F.max_pool1d(x, self.kernel_size, self.stride)
        if mask is None:
            return x_p, None
        m_p = F.max_pool1d(mask.float().unsqueeze(1), self.kernel_size, self.stride).squeeze(1) > 0
        return x_p, m_p


def masked_avg_pool(z: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
    if mask is None:
        return z.mean(-1)
    m = mask.unsqueeze(1).float()
    return (z * m).sum(-1) / m.sum(-1).clamp_min(1.0)

## 4. CNN tower (lite)

Changes vs v4.3:
- Width 128 → 64.
- Kernels (7, 15, 21) → (7, 15).
- 4 dilated blocks → 3 (drop dilation=8).
- Dropout 0.15 → 0.30, switched to channel-wise `Dropout1d` in `ConvBlock`.
- 4 sinusoidal positional-encoding channels appended to the one-hot input (carried from v4.4) so the trunk is no longer strictly translation-equivariant.
- Removed the RC *early-fusion* path entirely; only late-fusion (forward + RC averaged) is supported.
- Removed the v4.4 boundary segmentation head.

In [4]:
class CNNTowerLite(nn.Module):
    def __init__(
        self,
        width: int = CNN_WIDTH,
        motif_kernels: Tuple[int, ...] = MOTIF_KERNELS,
        context_kernel: int = CONTEXT_KERNEL,
        context_dilations: Tuple[int, ...] = CONTEXT_DILATIONS,
        pos_enc_channels: int = POS_ENC_CHANNELS,
        dropout: float = CNN_DROPOUT,
    ):
        super().__init__()
        self.out_dim = width
        self.pos_enc_channels = pos_enc_channels
        in_ch_per_motif = 5 + pos_enc_channels   # one-hot ACGTN + sinusoidal PE

        self.motif_convs = nn.ModuleList([
            nn.Sequential(
                nn.Conv1d(in_ch_per_motif, width, kernel_size=k, padding=k // 2, bias=True),
                nn.BatchNorm1d(width),
                nn.GELU(),
                nn.Dropout1d(dropout),
            )
            for k in motif_kernels
        ])

        self.mix = nn.Sequential(
            nn.Conv1d(width * len(motif_kernels), width, kernel_size=1, bias=True),
            nn.BatchNorm1d(width),
            nn.GELU(),
            nn.Dropout1d(dropout),
        )

        self.context_blocks = nn.ModuleList([
            ConvBlock(width, width, kernel_size=context_kernel, dilation=d, dropout=dropout)
            for d in context_dilations
        ])
        self.pool = MaskedMaxPool1d(kernel_size=2, stride=2)

        # Cache positional encoding lazily (built on first forward, then reused).
        self.register_buffer('_pe_cache', torch.zeros(0), persistent=False)

    # ---- positional encoding ----
    def _build_pos_enc(self, length: int, device: torch.device) -> torch.Tensor:
        """Return (pos_enc_channels, length) sinusoidal PE with log-spaced frequencies."""
        pe_dim = self.pos_enc_channels
        pos = torch.linspace(0.0, 1.0, length, device=device)            # (L,) in [0,1]
        # Frequencies: 1, 2, 4, ... cycles across the full canvas.
        freqs = (2.0 ** torch.arange(pe_dim // 2, device=device)) * math.pi
        ang = pos.unsqueeze(0) * freqs.unsqueeze(1)                      # (pe_dim/2, L)
        pe = torch.cat([torch.sin(ang), torch.cos(ang)], dim=0)          # (pe_dim, L)
        return pe

    def _add_pos_enc(self, x: torch.Tensor) -> torch.Tensor:
        """Append positional channels to one-hot input. x: (B, 5, L) -> (B, 5+pe, L)."""
        if self.pos_enc_channels == 0:
            return x
        B, _, L = x.shape
        if self._pe_cache.numel() == 0 or self._pe_cache.shape[-1] != L or self._pe_cache.device != x.device:
            self._pe_cache = self._build_pos_enc(L, x.device)
        pe = self._pe_cache.unsqueeze(0).expand(B, -1, -1)               # (B, pe, L)
        return torch.cat([x, pe], dim=1)

    # ---- RC transform ----
    @staticmethod
    def rc_transform(x: torch.Tensor, mask: torch.Tensor):
        x_rc = x.index_select(1, REV_COMP.to(x.device)).flip(-1)
        mask_rc = None if mask is None else mask.flip(-1)
        return x_rc, mask_rc

    # ---- single-pass encoding ----
    def encode(self, x: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        x = self._add_pos_enc(x)
        feats = [conv(x) for conv in self.motif_convs]
        z = self.mix(torch.cat(feats, dim=1))
        m = mask
        for block in self.context_blocks:
            z = block(z)
            z, m = self.pool(z, m)
        return masked_avg_pool(z, m)

    def forward(self, x: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        # Late-fusion RC averaging (v4.3 default).
        f = self.encode(x, mask)
        x_rc, mask_rc = self.rc_transform(x, mask)
        r = self.encode(x_rc, mask_rc)
        return 0.5 * (f + r)

## 5. GNN tower (lite)

Changes vs v4.3:
- Hidden 128 → 64.
- Layers 3 → 2.
- Input k-mer hash 7-mer/2048-d → 6-mer/1024-d (handled by the featurizer in `02_train.ipynb`; this module just consumes the resulting input dim).
- Dropout 0.10 → 0.25.

In [5]:
def scatter_mean(x: torch.Tensor, idx: torch.Tensor, dim_size: int) -> torch.Tensor:
    out = torch.zeros((dim_size, x.size(1)), device=x.device, dtype=x.dtype)
    out.index_add_(0, idx, x)
    cnt = torch.bincount(idx, minlength=dim_size).clamp_min(1).to(x.device).to(x.dtype).unsqueeze(1)
    return out / cnt


class GraphSAGELayer(nn.Module):
    def __init__(self, in_dim: int, out_dim: int, dropout: float = 0.1):
        super().__init__()
        self.lin_self = nn.Linear(in_dim, out_dim)
        self.lin_neigh = nn.Linear(in_dim, out_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
        src, dst = edge_index[0], edge_index[1]
        agg = torch.zeros_like(x)
        agg.index_add_(0, dst, x[src])
        deg = torch.bincount(dst, minlength=x.size(0)).clamp_min(1).to(x.device).to(x.dtype).unsqueeze(1)
        agg = agg / deg
        h = self.lin_self(x) + self.lin_neigh(agg)
        return self.dropout(F.relu(h))


class GNNTowerLite(nn.Module):
    def __init__(
        self,
        in_dim: int = GNN_IN_DIM,
        hidden: int = GNN_HIDDEN,
        n_layers: int = GNN_LAYERS,
        dropout: float = GNN_DROPOUT,
    ):
        super().__init__()
        self.out_dim = hidden
        layers = []
        d = in_dim
        for _ in range(n_layers):
            layers.append(GraphSAGELayer(d, hidden, dropout=dropout))
            d = hidden
        self.layers = nn.ModuleList(layers)

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor, batch_vec: torch.Tensor) -> torch.Tensor:
        for layer in self.layers:
            x = layer(x, edge_index)
        B = int(batch_vec.max().item()) + 1 if batch_vec.numel() else 0
        return scatter_mean(x, batch_vec, dim_size=B)

## 6. Cross-modal attention fusion (lite)

Changes vs v4.3:
- `fusion_dim` 256 → 128.
- `num_heads` 4 → 2.
- Dropout 0.20 → 0.30.
- Learned gate (CNN vs GNN) is **kept** — gate weights are informative for interpretation.

In [6]:
class CrossModalAttentionFusionLite(nn.Module):
    def __init__(
        self,
        cnn_dim: int = CNN_WIDTH,
        gnn_dim: int = GNN_HIDDEN,
        fusion_dim: int = FUSION_DIM,
        num_heads: int = NUM_HEADS,
        dropout: float = FUSION_DROPOUT,
    ):
        super().__init__()
        self.fusion_dim = fusion_dim
        self.cnn_proj = nn.Linear(cnn_dim, fusion_dim)
        self.gnn_proj = nn.Linear(gnn_dim, fusion_dim)
        self.ln1 = nn.LayerNorm(fusion_dim)
        self.ln2 = nn.LayerNorm(fusion_dim)
        self.cross_attn = nn.MultiheadAttention(
            embed_dim=fusion_dim, num_heads=num_heads, dropout=dropout, batch_first=True,
        )
        self.gate = nn.Sequential(
            nn.Linear(fusion_dim * 2, fusion_dim),
            nn.GELU(),
            nn.Linear(fusion_dim, 2),
            nn.Softmax(dim=-1),
        )
        self.out_proj = nn.Sequential(
            nn.Linear(fusion_dim, fusion_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )

    def forward(self, cnn_embed: torch.Tensor, gnn_embed: torch.Tensor):
        c = self.ln1(self.cnn_proj(cnn_embed))
        g = self.ln2(self.gnn_proj(gnn_embed))
        combined = torch.stack([c, g], dim=1)                  # (B, 2, F)
        attn_out, _ = self.cross_attn(combined, combined, combined)
        c_attn, g_attn = attn_out[:, 0], attn_out[:, 1]
        gate_weights = self.gate(torch.cat([c_attn, g_attn], dim=-1))   # (B, 2)
        fused = gate_weights[:, 0:1] * c_attn + gate_weights[:, 1:2] * g_attn
        return self.out_proj(fused), gate_weights

## 7. Full classifier

Changes vs v4.3:
- Two heads only: top-level (`num_toplevel`-way) and superfamily. **No** boundary regressor (v4.2/v4.3) and **no** segmentation head (v4.4).
- Head MLP widths shrunk to match the smaller fusion dim.
- The same class supports the binary variant by setting `num_toplevel=2`.

In [7]:
class HybridTEClassifierV43Lite(nn.Module):
    def __init__(
        self,
        num_toplevel: int = DEFAULT_NUM_TOPLEVEL,
        num_superfamilies: int = DEFAULT_NUM_SUPERFAMILIES,
        # CNN
        cnn_width: int = CNN_WIDTH,
        motif_kernels: Tuple[int, ...] = MOTIF_KERNELS,
        context_dilations: Tuple[int, ...] = CONTEXT_DILATIONS,
        pos_enc_channels: int = POS_ENC_CHANNELS,
        cnn_dropout: float = CNN_DROPOUT,
        # GNN
        gnn_in_dim: int = GNN_IN_DIM,
        gnn_hidden: int = GNN_HIDDEN,
        gnn_layers: int = GNN_LAYERS,
        gnn_dropout: float = GNN_DROPOUT,
        # Fusion
        fusion_dim: int = FUSION_DIM,
        num_heads: int = NUM_HEADS,
        fusion_dropout: float = FUSION_DROPOUT,
        # Heads
        head_dropout: float = 0.30,
    ):
        super().__init__()
        self.num_toplevel = num_toplevel
        self.num_superfamilies = num_superfamilies

        self.cnn_tower = CNNTowerLite(
            width=cnn_width,
            motif_kernels=motif_kernels,
            context_dilations=context_dilations,
            pos_enc_channels=pos_enc_channels,
            dropout=cnn_dropout,
        )
        self.gnn_tower = GNNTowerLite(
            in_dim=gnn_in_dim,
            hidden=gnn_hidden,
            n_layers=gnn_layers,
            dropout=gnn_dropout,
        )
        self.fusion = CrossModalAttentionFusionLite(
            cnn_dim=cnn_width,
            gnn_dim=gnn_hidden,
            fusion_dim=fusion_dim,
            num_heads=num_heads,
            dropout=fusion_dropout,
        )

        head_hidden = fusion_dim                               # 128
        self.toplevel_head = nn.Sequential(
            nn.Linear(fusion_dim, head_hidden),
            nn.GELU(),
            nn.Dropout(head_dropout),
            nn.Linear(head_hidden, num_toplevel),
        )
        self.superfamily_head = nn.Sequential(
            nn.Linear(fusion_dim, head_hidden),
            nn.GELU(),
            nn.Dropout(head_dropout),
            nn.Linear(head_hidden, num_superfamilies),
        )

    def encode(self, x_cnn, mask, x_gnn, edge_index, batch_vec):
        c = self.cnn_tower(x_cnn, mask)
        g = self.gnn_tower(x_gnn, edge_index, batch_vec)
        fused, gate_weights = self.fusion(c, g)
        return fused, gate_weights

    def forward(self, x_cnn, mask, x_gnn, edge_index, batch_vec, mixup_lam=None, mixup_perm=None):
        """If `mixup_lam` and `mixup_perm` are given, MixUp is applied on the fused embedding
        (top-level head only). The SF head sees the un-mixed embedding to keep fine-grained labels intact.
        """
        fused, gate_weights = self.encode(x_cnn, mask, x_gnn, edge_index, batch_vec)

        if mixup_lam is not None and mixup_perm is not None:
            fused_mix = mixup_lam * fused + (1.0 - mixup_lam) * fused[mixup_perm]
            top_logits = self.toplevel_head(fused_mix)
        else:
            top_logits = self.toplevel_head(fused)
        sf_logits = self.superfamily_head(fused)
        return top_logits, sf_logits, gate_weights

## 8. Parameter count check

Target: ~0.4–0.5 M trainable parameters (vs ~1.9 M in v4.3). The `assert` will fail loudly if the slim-down regressed.

In [8]:
def count_params(module: nn.Module):
    total = sum(p.numel() for p in module.parameters() if p.requires_grad)
    by_name = {name: sum(p.numel() for p in m.parameters() if p.requires_grad)
               for name, m in module.named_children()}
    return total, by_name

model = HybridTEClassifierV43Lite(
    num_toplevel=DEFAULT_NUM_TOPLEVEL,
    num_superfamilies=DEFAULT_NUM_SUPERFAMILIES,
).to(DEVICE)

total, by_name = count_params(model)
print(f'Total trainable params: {total:,} ({total/1e6:.3f} M)')
for name, n in by_name.items():
    print(f'  {name:<22s}{n:>10,}  ({100*n/total:5.1f}%)')

assert 250_000 <= total <= 700_000, (
    f'Lite model param count {total:,} is outside the target range [250k, 700k]; '
    'check the slim-down config in section 1.'
)
print('\nParam count within target range.')

Total trainable params: 441,500 (0.442 M)
  cnn_tower                132,608  ( 30.0%)
  gnn_tower                139,648  ( 31.6%)
  fusion                   132,866  ( 30.1%)
  toplevel_head             16,899  (  3.8%)
  superfamily_head          19,479  (  4.4%)

Param count within target range.


## 9. Smoke forward pass

Build a synthetic batch matching the shapes the real `collate_hybrid` will produce, run a forward pass, and assert the output shapes. This catches:
- mismatched channel counts after positional encoding;
- mis-sized fusion projections;
- batch_vec / scatter mismatches in the GNN.

The MixUp branch is also exercised.

In [9]:
B = SMOKE_BATCH
L = FIXED_LENGTH
N_per = SMOKE_NODES_PER_GRAPH

# --- CNN inputs: random one-hot sequences with random per-sample length ---
x_cnn = torch.zeros(B, 5, L)
mask  = torch.zeros(B, L, dtype=torch.bool)
for i in range(B):
    seq_len = int(torch.randint(2_000, L, (1,)).item())
    start = int(torch.randint(0, L - seq_len + 1, (1,)).item())
    bases = torch.randint(0, 4, (seq_len,))           # ACGT only (no N)
    x_cnn[i, bases, torch.arange(seq_len) + start] = 1.0
    mask[i, start:start + seq_len] = True
x_cnn = x_cnn.to(DEVICE); mask = mask.to(DEVICE)

# --- GNN inputs: random k-mer features with chain edges + self-loops ---
node_feats = []
edges      = []
batch_vec  = []
node_offset = 0
for i in range(B):
    n = N_per
    node_feats.append(torch.randn(n, GNN_IN_DIM))
    src = torch.arange(n - 1)
    dst = torch.arange(1, n)
    chain = torch.stack([torch.cat([src, dst, torch.arange(n)]),
                         torch.cat([dst, src, torch.arange(n)])], dim=0)
    edges.append(chain + node_offset)
    batch_vec.append(torch.full((n,), i, dtype=torch.long))
    node_offset += n
x_gnn      = torch.cat(node_feats, dim=0).to(DEVICE)
edge_index = torch.cat(edges, dim=1).to(DEVICE)
batch_vec  = torch.cat(batch_vec, dim=0).to(DEVICE)

# --- forward pass (no mixup) ---
model.eval()
with torch.no_grad():
    top, sf, gates = model(x_cnn, mask, x_gnn, edge_index, batch_vec)
print('Forward (no mixup):')
print(f'  top_logits   {tuple(top.shape)}   expected ({B}, {DEFAULT_NUM_TOPLEVEL})')
print(f'  sf_logits    {tuple(sf.shape)}    expected ({B}, {DEFAULT_NUM_SUPERFAMILIES})')
print(f'  gate_weights {tuple(gates.shape)} expected ({B}, 2)  sum={gates.sum(-1).mean().item():.3f}')
assert top.shape   == (B, DEFAULT_NUM_TOPLEVEL)
assert sf.shape    == (B, DEFAULT_NUM_SUPERFAMILIES)
assert gates.shape == (B, 2)

# --- forward pass with MixUp on the top-level head ---
model.train()
lam = 0.7
perm = torch.randperm(B, device=DEVICE)
top_m, sf_m, gates_m = model(x_cnn, mask, x_gnn, edge_index, batch_vec,
                              mixup_lam=lam, mixup_perm=perm)
assert top_m.shape == (B, DEFAULT_NUM_TOPLEVEL)
assert sf_m.shape  == (B, DEFAULT_NUM_SUPERFAMILIES)
print('\nForward (with MixUp on top-level): shapes OK.')

# --- backward pass to confirm grads flow through both towers ---
loss = F.cross_entropy(top_m, torch.zeros(B, dtype=torch.long, device=DEVICE)) \
     + F.cross_entropy(sf_m,  torch.zeros(B, dtype=torch.long, device=DEVICE))
loss.backward()
cnn_grad = next(model.cnn_tower.parameters()).grad
gnn_grad = next(model.gnn_tower.parameters()).grad
assert cnn_grad is not None and cnn_grad.abs().sum() > 0, 'CNN tower received no gradient'
assert gnn_grad is not None and gnn_grad.abs().sum() > 0, 'GNN tower received no gradient'
print(f'\nBackward OK. loss={loss.item():.4f}')
print('Smoke test passed — model is ready for the data pipeline in 02_train.ipynb.')

Forward (no mixup):
  top_logits   (4, 3)   expected (4, 3)
  sf_logits    (4, 23)    expected (4, 23)
  gate_weights (4, 2) expected (4, 2)  sum=1.000



Forward (with MixUp on top-level): shapes OK.



Backward OK. loss=4.1757
Smoke test passed — model is ready for the data pipeline in 02_train.ipynb.
